## Problem 1: Euclidean vs. cosine on the unit sphere

**What we're proving:** when two vectors both have length 1, the squared distance between them depends only on the angle between them.

**Step 1: expand the distance.**

$$\|x - y\|_2^2 = \|x\|_2^2 - 2\langle x, y\rangle + \|y\|_2^2$$

This is just $(a - b)^2 = a^2 - 2ab + b^2$, written with vectors.

**Step 2: use length 1.**

Both lengths are 1, so $\|x\|_2^2 = \|y\|_2^2 = 1$.

Cosine similarity is $\cos\theta = \dfrac{\langle x, y\rangle}{\|x\|_2\,\|y\|_2}$. With lengths of 1 the denominator is 1, so $\cos\theta = \langle x, y\rangle$.

**Step 3: substitute.**

$$\|x - y\|_2^2 = 1 - 2\cos\theta + 1 = 2(1 - \cos\theta) \qquad \blacksquare$$

### Why the neighbour rankings are the same

From the formula, distance $= \sqrt{2(1 - \cos\theta)}$.

As $\cos\theta$ goes up, $1 - \cos\theta$ goes down, so the distance goes down. Higher cosine similarity always means smaller distance, and the relationship never flips (the function is strictly decreasing).

So ranking neighbours by "smallest Euclidean distance" or by "largest cosine similarity" gives exactly the same order and the same $k$ nearest neighbours. Only the scale is different.

| Vectors point... | $\cos\theta$ | Distance |
|---|---|---|
| the same way | 1 | 0 |
| at 90° | 0 | $\sqrt{2}$ |
| opposite ways | −1 | 2 |

## Problem 2: Cosine distance is not a metric

**Definition:** $d_{\cos}(x, y) = 1 - \cos\theta(x, y)$.

A metric needs four things:
1. $d(x, y) \ge 0$
2. $d(x, y) = 0$ only when $x = y$
3. $d(x, y) = d(y, x)$
4. **Triangle inequality:** $d(x, z) \le d(x, y) + d(y, z)$

Cosine distance passes the first three on the unit sphere, but it **fails the triangle inequality**.

### Counterexample in $\mathbb{R}^2$

Take three unit vectors at 0°, 45° and 90°:

$$x = (1, 0), \qquad y = \left(\tfrac{1}{\sqrt{2}}, \tfrac{1}{\sqrt{2}}\right), \qquad z = (0, 1)$$

Compute the cosines (all vectors have length 1, so cosine is just the dot product):

| Pair | Angle | $\cos\theta$ | $d_{\cos} = 1 - \cos\theta$ |
|---|---|---|---|
| $x, y$ | 45° | $\tfrac{\sqrt{2}}{2} \approx 0.707$ | $\approx 0.293$ |
| $y, z$ | 45° | $\tfrac{\sqrt{2}}{2} \approx 0.707$ | $\approx 0.293$ |
| $x, z$ | 90° | $0$ | $1$ |

Check the triangle inequality:

$$d_{\cos}(x, z) = 1 \quad > \quad d_{\cos}(x, y) + d_{\cos}(y, z) \approx 0.586$$

Going "through" $y$ is shorter than going straight from $x$ to $z$, which is impossible for a real distance. So $d_{\cos}$ is **not a metric**. $\blacksquare$

**Why it breaks:** from Problem 1, on the unit sphere $d_{\cos}(x, y) = \tfrac{1}{2}\|x - y\|_2^2$. That's a *squared* Euclidean distance, and squaring breaks the triangle inequality (e.g. on a line, $0 \to 1 \to 2$: $2^2 = 4 > 1^2 + 1^2 = 2$). For small angles, $1 - \cos\theta \approx \theta^2/2$, so small angles get shrunk too much compared to big ones.

### Contrast: angular distance

$$d_{\text{ang}}(x, y) = \frac{\arccos(\cos\theta)}{\pi} = \frac{\theta}{\pi} \in [0, 1]$$

This just measures the angle itself, scaled to $[0, 1]$. Same example:

| Pair | Angle | $d_{\text{ang}}$ |
|---|---|---|
| $x, y$ | 45° | $\tfrac{1}{4}$ |
| $y, z$ | 45° | $\tfrac{1}{4}$ |
| $x, z$ | 90° | $\tfrac{1}{2}$ |

$$d_{\text{ang}}(x, z) = \tfrac{1}{2} \quad \le \quad \tfrac{1}{4} + \tfrac{1}{4} = \tfrac{1}{2} \quad \checkmark$$

The triangle inequality holds (here with equality, since $y$ lies exactly between $x$ and $z$).

**Why angular distance is a metric:** the angle between two points on the unit sphere is the length of the shortest path along the sphere's surface (the great-circle arc). Shortest-path lengths always satisfy the triangle inequality, because a detour through $y$ can never beat the direct path. Dividing by $\pi$ just rescales it.

*Small note:* on vectors that aren't normalised, $x$ and $2x$ have angle 0, so $d_{\text{ang}}$ is only a metric on directions (i.e. on the unit sphere), not on raw vectors.

### Summary

| | $d_{\cos} = 1 - \cos\theta$ | $d_{\text{ang}} = \theta / \pi$ |
|---|---|---|
| Triangle inequality | ✗ fails | ✓ holds |
| Metric? | No | Yes (on the sphere) |
| Neighbour ranking | Same as angle | Same as angle |

Both are increasing functions of $\theta$, so they **rank neighbours the same way**. The difference only matters when you need actual metric properties, e.g. for metric-tree indexes (ball trees, VP-trees) or triangle-inequality pruning, which can give wrong results with $d_{\cos}$.

## Problem 3: Curse of dimensionality

### Part A: in high dimensions, all points are about equally far away

**Setup:** draw $n$ points $X^{(1)}, \dots, X^{(n)}$ uniformly from the cube $[0,1]^d$ and fix a query point $q$. Let $D_{\min}$ and $D_{\max}$ be the distances from $q$ to its nearest and farthest point. We want to show

$$\frac{D_{\max} - D_{\min}}{D_{\min}} \to 0 \quad \text{as } d \to \infty.$$

**Step 1: write the squared distance as a sum.**

For one random point $X$:

$$D^2 = \|X - q\|_2^2 = \sum_{j=1}^{d} \underbrace{(X_j - q_j)^2}_{Z_j}$$

The coordinates $X_j$ are independent, so $D^2$ is a sum of $d$ **independent** terms $Z_j$, each between 0 and 1.

**Step 2: the mean grows like $d$, the spread grows like $\sqrt{d}$.**

- Each $Z_j$ has mean at least $\tfrac{1}{12}$ (the smallest case, when $q_j = \tfrac12$). So the mean is $\mu = \mathbb{E}[D^2] \ge \tfrac{d}{12}$, which grows **linearly** in $d$.
- Each $Z_j$ lies in $[0, 1]$, so $\operatorname{Var}(Z_j) \le \tfrac14$. Independent variances add, so $\operatorname{Var}(D^2) \le \tfrac{d}{4}$, and the standard deviation is at most $\tfrac{\sqrt{d}}{2}$, which grows only like **$\sqrt{d}$**.

So the relative spread is

$$\frac{\text{std}(D^2)}{\mathbb{E}[D^2]} \le \frac{\sqrt{d}/2}{d/12} = \frac{6}{\sqrt{d}} \to 0.$$

The distance gets bigger, but its random wiggle gets **relatively** smaller. This is the law of large numbers at work.

**Step 3: make it precise with Chebyshev's inequality.**

For any $\varepsilon > 0$ and one point:

$$P\big(|D^2 - \mu| \ge \varepsilon\mu\big) \le \frac{\operatorname{Var}(D^2)}{\varepsilon^2 \mu^2} \le \frac{d/4}{\varepsilon^2 d^2/144} = \frac{36}{\varepsilon^2 d}.$$

For all $n$ points at once (union bound):

$$P\big(\text{some point has } |D^2 - \mu| \ge \varepsilon\mu\big) \le \frac{36\,n}{\varepsilon^2 d} \to 0 \quad (n \text{ fixed}, \ d \to \infty).$$

So with probability going to 1, **every** squared distance is in $[(1-\varepsilon)\mu,\ (1+\varepsilon)\mu]$. Then

$$\frac{D_{\max}}{D_{\min}} \le \sqrt{\frac{1+\varepsilon}{1-\varepsilon}} \quad\Longrightarrow\quad \frac{D_{\max} - D_{\min}}{D_{\min}} \le \sqrt{\frac{1+\varepsilon}{1-\varepsilon}} - 1.$$

The right side can be made as small as we like by picking a small $\varepsilon$, so $\dfrac{D_{\max} - D_{\min}}{D_{\min}} \to 0$ in probability. $\blacksquare$

**Intuition:** $D$ grows like $\sqrt{d}$, but the gap $D_{\max} - D_{\min}$ stays roughly constant. A constant gap divided by something growing like $\sqrt{d}$ goes to 0, at a rate of about $1/\sqrt{d}$. The nearest and farthest points end up at almost the same distance.

### Part B: almost all of a high-dimensional ball is near its surface

**Step 1: volume scales like $r^d$.**

A $d$-dimensional ball of radius $r$ has volume $V_d(r) = C_d\, r^d$, where $C_d$ is a constant depending only on $d$.

**Step 2: compute the shell fraction.**

The shell of thickness $\varepsilon$ is the unit ball minus the inner ball of radius $1 - \varepsilon$:

$$\frac{V_d(1) - V_d(1-\varepsilon)}{V_d(1)} = \frac{C_d - C_d(1-\varepsilon)^d}{C_d} = 1 - (1-\varepsilon)^d.$$

**Step 3: take the limit.**

Since $0 < 1 - \varepsilon < 1$, raising it to a huge power sends it to 0. More precisely, $(1-\varepsilon)^d \le e^{-\varepsilon d} \to 0$. So

$$1 - (1-\varepsilon)^d \to 1. \qquad \blacksquare$$

**How fast?** Fraction of volume in the outer shell:

| $d$ | $\varepsilon = 0.01$ (outer 1%) | $\varepsilon = 0.1$ (outer 10%) |
|---|---|---|
| 2 | 2.0% | 19% |
| 10 | 9.6% | 65% |
| 100 | 63% | 99.997% |
| 1000 | 99.996% | ≈ 100% |

In 1000 dimensions, a skin only 1% thick holds essentially all the volume. The "inside" of the ball is basically empty.

### Consequences for nearest-neighbour search

1. **"Nearest" stops meaning much.** If the nearest point is only slightly closer than the farthest one (Part A), the ranking is fragile: a bit of noise in the features can reorder the neighbours completely.
2. **Exact indexes stop helping.** k-d trees and ball trees skip regions using distance bounds ("this whole box is farther than my current best"). When all distances are almost equal, nothing can be skipped, and the search falls back to checking every point. k-d trees typically degrade beyond about 10–20 dimensions.
3. **Data is spread thin.** Since volume concentrates at the boundary (Part B) and grows like $r^d$, covering the space so that every query has a genuinely close neighbour needs a number of points that grows **exponentially** in $d$.
4. **Cosine is affected too.** Random high-dimensional vectors are nearly orthogonal ($\cos\theta \approx 0 \pm 1/\sqrt{d}$). By Problem 1, cosine and Euclidean rank the same way on the sphere, so switching metric does not escape the problem.

**Why NN search still works on real data:** real data (images, text embeddings) is **not** uniform in $[0,1]^d$. It sits near a low-dimensional structure (its *intrinsic dimension* is small), so distances stay informative. This motivates:
- **dimensionality reduction**, e.g. PCA (Problem 4), to keep only the directions that carry signal;
- **learned embeddings**, which put similar items close together on purpose;
- **approximate NN** methods (HNSW, IVF, LSH), which give up exactness for speed in high dimensions.